In [ ]:
!pip install -q nlp-id
# nlp-id pins an old huggingface_hub that breaks recent transformers -> upgrade it back
!pip install -q -U huggingface_hub
# If Colab asks to restart the session after this cell, restart and run all again.

In [ ]:
import os
os.environ['NLTK_ALLOW_PROXIED_URLOPEN'] = '1'  # newer NLTK blocks downloads through Colab's proxy without this
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
from nltk.tokenize import word_tokenize, sent_tokenize

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────
# Put scores.pkl, question_ids.pkl, preprocessed_texts.pkl and
# preprocessed_sbert_texts.pkl in DATA_DIR (Google Drive when using Colab).
DATA_DIR    = "/content/drive/MyDrive/aes_data/"
ESSAY_PATH  = DATA_DIR
SCORE_PATH  = DATA_DIR
OUTPUT_PATH = "/content/drive/MyDrive/aes_results/"

---

# FA - ML

In [ ]:
"""
=============================================================
  AES — Paradigma B: SBERT Hybrid
  Metode   : Frozen embedding + Feature Engineering + XGBoost
  Dataset  : preprocessed_texts.pkl, scores.pkl
  Model    : denaya/indoSBERT-large
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation (KFold)
  Referensi: Faseeh et al. (2024), Chen & Guestrin (2016)
=============================================================
"""

import os
import time
import threading
import traceback
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error
from nltk.tokenize import word_tokenize
from nlp_id.postag import PosTag
import xgboost as xgb
import pynvml



# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay": f"{ESSAY_PATH}preprocessed_sbert_texts.pkl",
    "score": f"{SCORE_PATH}scores.pkl",
    "model_name"   : "denaya/indoSBERT-large",
    "max_len"      : 256,
    "embed_dim"    : 256,
    "batch_size"   : 32,
    "xgb_n_estimators"     : 300,
    "xgb_learning_rate"    : 0.05,
    "xgb_max_depth"        : 3,
    "xgb_subsample"        : 0.5,
    "xgb_colsample_bytree" : 0.5,
    "xgb_reg_lambda"       : 2.0,
    "xgb_reg_alpha"        : 0.1,
    "xgb_gamma"            : 0.1,
    "xgb_min_child_weight" : 20,
    "xgb_early_stop"       : 20,
    "n_folds"      :10,
    "seed"         : 42,
    "output_dir"   : f"{OUTPUT_PATH}output_sbert_hyb",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Model : {CFG['model_name']} (FROZEN)")


# ══════════════════════════════════════════════════════════
#  PYNVML — polling peak GPU untuk XGBoost (non-PyTorch)
# ══════════════════════════════════════════════════════════
def _gpu_poll_worker(stop_event, peak_holder, device_index=0):
    """Thread worker: poll VRAM setiap 50ms, simpan nilai tertinggi."""
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
    peak = 0
    while not stop_event.is_set():
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if info.used > peak:
            peak = info.used
        stop_event.wait(timeout=0.05)
    peak_holder.append(peak)
    pynvml.nvmlShutdown()


def measure_gpu_peak_mb(fn, device_index=0):
    """
    Jalankan fn() sambil polling VRAM via pynvml.
    Return: (result_of_fn, peak_mb)
    """
    stop_event  = threading.Event()
    peak_holder = []
    t = threading.Thread(
        target=_gpu_poll_worker,
        args=(stop_event, peak_holder, device_index),
        daemon=True,
    )
    t.start()
    result = fn()
    stop_event.set()
    t.join()
    peak_mb = peak_holder[0] / (1024 ** 2) if peak_holder else 0.0
    return result, peak_mb


# ══════════════════════════════════════════════════════════
#  LINGUISTIC FEATURE EXTRACTION
#  Fitur:
#  1. ttr              — lexical diversity (unique/total tokens)
#  2. word_count       — panjang respons, proxy cakupan konten
#  3. noun_count       — kepadatan konten faktual (NN, NNP)
#  4. verb_count       — elaborasi naratif/kausal (VB)
#  5. sentence_count   — kompleksitas sintaksis (proxy)
#  6. avg_word_length  — kompleksitas morfologi (proxy, Indonesian-specific)
#
#  Library:
#  - nltk.word_tokenize  : tokenisasi kata
#  - nltk.sent_tokenize  : deteksi batas kalimat
#  - nlp_id.postag       : POS tagging bahasa Indonesia
# ══════════════════════════════════════════════════════════
postag = PosTag()

POS_NOUN = {'NN', 'NNP'}
POS_VERB = {'VB'}

FEATURE_NAMES = [
    'ttr',
    'word_count',
    'noun_count',
    'verb_count',
    'sentence_count',
    'avg_word_length',
]


def extract_linguistic_features(text: str) -> dict:
    """
    Args:
        text : teks preprocessed (output preprocess_text)

    Returns:
        dict dengan 6 fitur (urutan sesuai FEATURE_NAMES)
    """
    if not text:
        return {f: 0.0 for f in FEATURE_NAMES}

    # --- Tokenisasi kata ---
    words = word_tokenize(text)
    # Filter: hanya kata alfanumerik (bukan tanda baca yang ikut ter-tokenize)
    word_tokens = [w for w in words if w.isalnum()]
    n_words = len(word_tokens)

    if n_words == 0:
        return {f: 0.0 for f in FEATURE_NAMES}

    # 1. TTR — type-token ratio
    ttr = len(set(word_tokens)) / n_words

    # 2. Word count
    word_count = float(n_words)

    # 3 & 4. POS counts (noun, verb) via nlp_id
    pos_tags   = postag.get_pos_tag(text)
    noun_count = sum(1 for _, pos in pos_tags if pos in POS_NOUN)
    verb_count = sum(1 for _, pos in pos_tags if pos in POS_VERB)

    # 5. Sentence count — valid karena tanda baca dipertahankan
    sentences      = sent_tokenize(text)
    sentence_count = float(max(len(sentences), 1))

    # 6. Average word length (characters per token)
    #    Proxy morphological complexity untuk bahasa Indonesia aglutinatif
    avg_word_length = sum(len(w) for w in word_tokens) / n_words

    return {
        'ttr'            : ttr,
        'word_count'     : word_count,
        'noun_count'     : float(noun_count),
        'verb_count'     : float(verb_count),
        'sentence_count' : sentence_count,
        'avg_word_length': avg_word_length,
    }


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating=None, max_rating=None):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)
    if min_rating is None: min_rating = min(y_true.min(), y_pred.min())
    if max_rating is None: max_rating = max(y_true.max(), y_pred.max())
    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1
    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored
    weights    = np.zeros((num_ratings, num_ratings))
    for i in range(num_ratings):
        for j in range(num_ratings):
            weights[i][j] = ((i - j) ** 2) / ((num_ratings - 1) ** 2)
    numerator   = np.sum(weights * conf_mat)
    denominator = np.sum(weights * expected)
    return 0.0 if denominator == 0 else 1.0 - numerator / denominator


# ══════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════
class EssayEmbedDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}


# ══════════════════════════════════════════════════════════
#  FROZEN EXTRACTOR
# ══════════════════════════════════════════════════════════
class SBERTExtractor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out      = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_emb = out.last_hidden_state
        mask_exp  = attention_mask.unsqueeze(-1).float()
        sum_emb   = torch.sum(token_emb * mask_exp, dim=1)
        sum_mask  = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
        return sum_emb / sum_mask


def extract_embeddings(texts, tokenizer, model, cfg, device):
    """
    Ekstrak embedding + ukur peak GPU via torch.
    Returns: embeddings, t_emb_total, gpu_mem_emb_mb
    """
    model.eval()
    dataset  = EssayEmbedDataset(texts, tokenizer, cfg["max_len"])
    loader   = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
    all_embs = []

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    t_start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            embs = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            all_embs.append(embs.cpu().numpy())
    t_total = time.perf_counter() - t_start

    gpu_mem_mb = (torch.cuda.max_memory_allocated(device) / (1024 ** 2)
                  if device.type == "cuda" else 0.0)

    return np.vstack(all_embs), t_total, gpu_mem_mb


# ══════════════════════════════════════════════════════════
#  XGBOOST
# ══════════════════════════════════════════════════════════
def train_xgboost_static(X_train, y_train, X_val, y_val, cfg):
    params = {
        "n_estimators"    : cfg["xgb_n_estimators"],
        "learning_rate"   : cfg["xgb_learning_rate"],
        "max_depth"       : cfg["xgb_max_depth"],
        "subsample"       : cfg["xgb_subsample"],
        "colsample_bytree": cfg["xgb_colsample_bytree"],
        "booster"         : "gbtree",
        "reg_lambda"      : cfg["xgb_reg_lambda"],
        "reg_alpha"       : cfg["xgb_reg_alpha"],
        "gamma"           : cfg["xgb_gamma"],
        "min_child_weight": cfg["xgb_min_child_weight"],
        "tree_method"     : "hist",
        "device"          : "cuda",
        "random_state"    : cfg["seed"],
        "verbosity"       : 0,
    }
    model = xgb.XGBRegressor(**params, early_stopping_rounds=cfg["xgb_early_stop"])
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model, params


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx, X_all, y_all, cfg, device, emb_time_per_fold, es_idx=None):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold_idx + 1}/{cfg['n_folds']}")
    print(f"{'='*55}")

    X_train = X_all[train_idx]; X_val = X_all[val_idx]
    y_train = y_all[train_idx]; y_val = y_all[val_idx]
    X_es, y_es = X_all[es_idx], y_all[es_idx]   # [FIX-LEAKAGE] early-stopping set

    print(f"  Train: {len(X_train)} | Val: {len(X_val)}")

    # ── XGBoost Training + peak GPU via pynvml ──
    gpu_index = device.index if (device.type == "cuda" and device.index is not None) else 0

    t_xgb_start = time.perf_counter()
    (final_model, best_params), xgb_gpu_peak_mb = measure_gpu_peak_mb(
        lambda: train_xgboost_static(X_train, y_train, X_es, y_es, cfg),
        device_index=gpu_index,
    )
    t_xgb_train = time.perf_counter() - t_xgb_start

    total_training_time = emb_time_per_fold + t_xgb_train

    # ── Inference per sampel (1 sampel, diulang untuk stabilitas) ──
    _ = final_model.predict(X_val[:1])   # warm-up
    t_infer_start = time.perf_counter()
    _ = final_model.predict(X_val[:1])
    inference_time_ms = (time.perf_counter() - t_infer_start) * 1000

    # ── Prediksi final ──
    preds_val = final_model.predict(X_val)
    mse_val   = mean_squared_error(y_val, preds_val)
    qwk_val   = quadratic_weighted_kappa(y_val, preds_val, cfg["score_min"], cfg["score_max"])

    print(f"\n  → Fold {fold_idx+1} FINAL:")
    print(f"    QWK                    = {qwk_val:.4f}")
    print(f"    MSE                    = {mse_val:.4f}")
    print(f"    training_time_s        = {total_training_time:.2f}s "
          f"(emb_share={emb_time_per_fold:.2f}s + xgb={t_xgb_train:.2f}s)")
    print(f"    inference_ms           = {inference_time_ms:.4f} ms/sampel")
    print(f"    memory_mb [xgb_gpu]    = {xgb_gpu_peak_mb:.2f} MB")

    fold_preds_df = pd.DataFrame({
        "true_score": y_val, "pred_score": preds_val, "fold": fold_idx + 1,
    })

    result = {
        "fold"            : fold_idx + 1,
        "qwk"             : qwk_val,
        "mse"             : mse_val,
        "training_time_s" : total_training_time,
        "inference_ms"    : inference_time_ms,
        "memory_mb_xgb_gpu" : xgb_gpu_peak_mb,
        "best_params"     : best_params,
    }
    return result, fold_preds_df, final_model


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    np.random.seed(CFG["seed"])

    print("\n[1/5] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])
    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())
    print(f"  Total sampel : {len(texts)}")
    print(f"  Rentang skor : {CFG['score_min']} – {CFG['score_max']}")

    print("\n[2/5] Mengekstrak fitur linguistik...")
    fe_list = []
    for i, text in enumerate(texts):
        if (i + 1) % 50 == 0 or i == 0: print(f"  [{i+1}/{len(texts)}] proses...")
        fe_list.append(extract_linguistic_features(text))
    features_raw = np.array([[d[k] for k in FEATURE_NAMES] for d in fe_list], dtype=np.float32)

    print(f"\n[3/5] Mengekstrak frozen embedding: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
    extractor = SBERTExtractor(CFG["model_name"])
    extractor.to(DEVICE).eval()

    embeddings, t_emb_total, gpu_mem_emb_mb = extract_embeddings(
        texts, tokenizer, extractor, CFG, DEVICE
    )
    emb_time_per_fold = t_emb_total / CFG["n_folds"]

    print(f"  Embedding shape        : {embeddings.shape}")
    print(f"  Waktu ekstraksi total  : {t_emb_total:.2f}s")
    print(f"  Waktu per fold (share) : {emb_time_per_fold:.2f}s")
    print(f"  Peak GPU memory (emb)  : {gpu_mem_emb_mb:.2f} MB")

    emb_path = os.path.join(CFG["output_dir"], "embeddings_sbert_hyb.npy")
    np.save(emb_path, embeddings)

    X_all = np.concatenate([embeddings, features_raw], axis=1).astype(np.float32)
    y_all = scores.astype(np.float32)
    print(f"\n  Input matrix X_all: {X_all.shape}")

    print(f"\n[4/5] Memulai {CFG['n_folds']}-fold cross-validation...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

    all_results = []; all_preds_dfs = []; fold_models = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):

        train_idx, es_idx = inner_split(train_idx, y_all, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df, fold_model = run_fold(
                fold_idx, train_idx, val_idx,
                X_all, y_all, CFG, DEVICE,
                emb_time_per_fold=emb_time_per_fold, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
            fold_models.append(fold_model)
            model_path = os.path.join(CFG["output_dir"], f"xgb_model_fold{fold_idx+1}_sbert_hyb.json")
            fold_model.save_model(model_path)
        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1} gagal: {e}")
            traceback.print_exc()

    print(f"\n{'='*55}")
    print(f"  RINGKASAN HASIL — SBERT Hybrid")
    print(f"{'='*55}")

    results_df   = pd.DataFrame(all_results)
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)

    cols_print = [c for c in results_df.columns if c != 'best_params']
    print(results_df[cols_print].to_string(index=False))

    print(f"\n  Rata-rata ± Std:")
    for metric in ["qwk", "mse", "training_time_s", "inference_ms", "memory_mb_xgb_gpu"]:
        m = results_df[metric].mean(); s = results_df[metric].std()
        print(f"    {metric:<28s}: {m:.4f} ± {s:.4f}")

    print(f"\n  Embedding extraction (global):")
    print(f"    peak_gpu_mem_emb_mb : {gpu_mem_emb_mb:.2f} MB")

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values, all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"],
    )
    overall_mse = mean_squared_error(all_preds_df["true_score"].values, all_preds_df["pred_score"].values)
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")

    experiment_data = {
        "config"                    : CFG,
        "fold_results"              : all_results,
        "fold_predictions"          : all_preds_df,
        "overall_qwk"               : overall_qwk,
        "overall_mse"               : overall_mse,
        "timestamp"                 : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "feature_names"             : FEATURE_NAMES,
        "embed_shape"               : embeddings.shape,
        "embedding_extraction_time_s" : t_emb_total,
        "embedding_time_per_fold_s" : emb_time_per_fold,
        "peak_gpu_mem_embedding_mb" : gpu_mem_emb_mb,
    }
    joblib_path = os.path.join(CFG["output_dir"], "experiment_results.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df[cols_print].to_csv(os.path.join(CFG["output_dir"], "cv_results_sbert_hyb.csv"), index=False)
    all_preds_df.to_csv(os.path.join(CFG["output_dir"], "all_predictions_sbert_hyb.csv"), index=False)

    print(f"\n  Hasil lengkap disimpan: {joblib_path}")
    return results_df


if __name__ == "__main__":
    main()

In [ ]:
"""
=============================================================
  AES — Paradigma B: IndoBERT Hybrid
  Metode   : Frozen embedding + Feature Engineering + XGBoost
  Dataset  : preprocessed_texts.pkl, scores.pkl
  Model    : indobenchmark/indobert-base-p2
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation (KFold)
  Referensi: Faseeh et al. (2024), Chen & Guestrin (2016)

=============================================================
"""

import os
import time
import threading
import traceback
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error
from nltk.tokenize import word_tokenize
from nlp_id.postag import PosTag
import xgboost as xgb
import pynvml


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay"        : f"{ESSAY_PATH}preprocessed_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",
    "model_name"   : "indobenchmark/indobert-base-p2",
    "max_len"      : 256,
    "embed_dim"    : 768,
    "batch_size"   : 64,
    "xgb_n_estimators"     : 300,
    "xgb_learning_rate"    : 0.05,
    "xgb_max_depth"        : 3,
    "xgb_subsample"        : 0.5,
    "xgb_colsample_bytree" : 0.5,
    "xgb_reg_lambda"       : 2.0,
    "xgb_reg_alpha"        : 0.1,
    "xgb_gamma"            : 0.1,
    "xgb_min_child_weight" : 20,
    "xgb_early_stop"       : 20,
    "n_folds"      : 10,
    "seed"         : 42,
    "output_dir"   : f"{OUTPUT_PATH}Hasil eksperiment/output_indobert_hyb",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Model : {CFG['model_name']} (FROZEN)")


# ══════════════════════════════════════════════════════════
#  PYNVML — polling peak GPU untuk XGBoost (non-PyTorch)
# ══════════════════════════════════════════════════════════
def _gpu_poll_worker(stop_event, peak_holder, device_index=0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
    peak = 0
    while not stop_event.is_set():
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if info.used > peak:
            peak = info.used
        stop_event.wait(timeout=0.05)
    peak_holder.append(peak)
    pynvml.nvmlShutdown()


def measure_gpu_peak_mb(fn, device_index=0):
    stop_event  = threading.Event()
    peak_holder = []
    t = threading.Thread(target=_gpu_poll_worker,
                         args=(stop_event, peak_holder, device_index), daemon=True)
    t.start()
    result = fn()
    stop_event.set()
    t.join()
    peak_mb = peak_holder[0] / (1024 ** 2) if peak_holder else 0.0
    return result, peak_mb


# ══════════════════════════════════════════════════════════
#  LINGUISTIC FEATURE EXTRACTION
#  Fitur:
#  1. ttr              — lexical diversity (unique/total tokens)
#  2. word_count       — panjang respons, proxy cakupan konten
#  3. noun_count       — kepadatan konten faktual (NN, NNP)
#  4. verb_count       — elaborasi naratif/kausal (VB)
#  5. sentence_count   — kompleksitas sintaksis (proxy)
#  6. avg_word_length  — kompleksitas morfologi (proxy, Indonesian-specific)
# ══════════════════════════════════════════════════════════
postag = PosTag()

POS_NOUN = {'NN', 'NNP'}
POS_VERB = {'VB'}

FEATURE_NAMES = [
    'ttr',
    'word_count',
    'noun_count',
    'verb_count',
    'sentence_count',
    'avg_word_length',
]


def extract_linguistic_features(text: str) -> dict:
    """
    Args:
        text : teks preprocessed (output preprocess_text)

    Returns:
        dict dengan 6 fitur (urutan sesuai FEATURE_NAMES)
    """
    if not text:
        return {f: 0.0 for f in FEATURE_NAMES}

    # --- Tokenisasi kata ---
    words = word_tokenize(text)
    # Filter: hanya kata alfanumerik (bukan tanda baca yang ikut ter-tokenize)
    word_tokens = [w for w in words if w.isalnum()]
    n_words = len(word_tokens)

    if n_words == 0:
        return {f: 0.0 for f in FEATURE_NAMES}

    # 1. TTR — type-token ratio
    ttr = len(set(word_tokens)) / n_words

    # 2. Word count
    word_count = float(n_words)

    # 3 & 4. POS counts (noun, verb) via nlp_id
    pos_tags   = postag.get_pos_tag(text)
    noun_count = sum(1 for _, pos in pos_tags if pos in POS_NOUN)
    verb_count = sum(1 for _, pos in pos_tags if pos in POS_VERB)

    # 5. Sentence count — valid karena tanda baca dipertahankan
    sentences      = sent_tokenize(text)
    sentence_count = float(max(len(sentences), 1))

    # 6. Average word length (characters per token)
    #    Proxy morphological complexity untuk bahasa Indonesia aglutinatif
    avg_word_length = sum(len(w) for w in word_tokens) / n_words

    return {
        'ttr'            : ttr,
        'word_count'     : word_count,
        'noun_count'     : float(noun_count),
        'verb_count'     : float(verb_count),
        'sentence_count' : sentence_count,
        'avg_word_length': avg_word_length,
    }


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating=None, max_rating=None):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)
    if min_rating is None: min_rating = min(y_true.min(), y_pred.min())
    if max_rating is None: max_rating = max(y_true.max(), y_pred.max())
    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1
    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored
    weights    = np.zeros((num_ratings, num_ratings))
    for i in range(num_ratings):
        for j in range(num_ratings):
            weights[i][j] = ((i - j) ** 2) / ((num_ratings - 1) ** 2)
    numerator   = np.sum(weights * conf_mat)
    denominator = np.sum(weights * expected)
    return 0.0 if denominator == 0 else 1.0 - numerator / denominator


# ══════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════
class EssayEmbedDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}


# ══════════════════════════════════════════════════════════
#  FROZEN EXTRACTOR — [CLS] token
# ══════════════════════════════════════════════════════════
class IndoBERTExtractor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]   # [CLS] dim 768


def extract_embeddings(texts, tokenizer, model, cfg, device):
    model.eval()
    dataset  = EssayEmbedDataset(texts, tokenizer, cfg["max_len"])
    loader   = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
    all_embs = []

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    t_start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            embs = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            all_embs.append(embs.cpu().numpy())
    t_total = time.perf_counter() - t_start

    gpu_mem_mb = (torch.cuda.max_memory_allocated(device) / (1024 ** 2)
                  if device.type == "cuda" else 0.0)

    return np.vstack(all_embs), t_total, gpu_mem_mb


# ══════════════════════════════════════════════════════════
#  XGBOOST
# ══════════════════════════════════════════════════════════
def train_xgboost_static(X_train, y_train, X_val, y_val, cfg):
    params = {
        "n_estimators"    : cfg["xgb_n_estimators"],
        "learning_rate"   : cfg["xgb_learning_rate"],
        "max_depth"       : cfg["xgb_max_depth"],
        "subsample"       : cfg["xgb_subsample"],
        "colsample_bytree": cfg["xgb_colsample_bytree"],
        "booster"         : "gbtree",
        "reg_lambda"      : cfg["xgb_reg_lambda"],
        "reg_alpha"       : cfg["xgb_reg_alpha"],
        "gamma"           : cfg["xgb_gamma"],
        "min_child_weight": cfg["xgb_min_child_weight"],
        "tree_method"     : "hist",
        "device"          : "cuda",
        "random_state"    : cfg["seed"],
        "verbosity"       : 0,
    }
    model = xgb.XGBRegressor(**params, early_stopping_rounds=cfg["xgb_early_stop"])
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model, params


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx, X_all, y_all, cfg, device, emb_time_per_fold, es_idx=None):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold_idx + 1}/{cfg['n_folds']}")
    print(f"{'='*55}")

    X_train = X_all[train_idx]; X_val = X_all[val_idx]
    y_train = y_all[train_idx]; y_val = y_all[val_idx]
    X_es, y_es = X_all[es_idx], y_all[es_idx]   # [FIX-LEAKAGE] early-stopping set
    print(f"  Train: {len(X_train)} | Val: {len(X_val)}")

    gpu_index = device.index if (device.type == "cuda" and device.index is not None) else 0

    t_xgb_start = time.perf_counter()
    (final_model, best_params), xgb_gpu_peak_mb = measure_gpu_peak_mb(
        lambda: train_xgboost_static(X_train, y_train, X_es, y_es, cfg),
        device_index=gpu_index,
    )
    t_xgb_train = time.perf_counter() - t_xgb_start

    total_training_time = emb_time_per_fold + t_xgb_train

    _ = final_model.predict(X_val[:1])   # warm-up
    t_infer_start = time.perf_counter()
    _ = final_model.predict(X_val[:1])
    inference_time_ms = (time.perf_counter() - t_infer_start) * 1000

    preds_val = final_model.predict(X_val)
    mse_val   = mean_squared_error(y_val, preds_val)
    qwk_val   = quadratic_weighted_kappa(y_val, preds_val, cfg["score_min"], cfg["score_max"])

    print(f"\n  → Fold {fold_idx+1} FINAL:")
    print(f"    QWK                    = {qwk_val:.4f}")
    print(f"    MSE                    = {mse_val:.4f}")
    print(f"    training_time_s        = {total_training_time:.2f}s "
          f"(emb_share={emb_time_per_fold:.2f}s + xgb={t_xgb_train:.2f}s)")
    print(f"    inference_ms           = {inference_time_ms:.4f} ms/sampel")
    print(f"    memory_mb [xgb_gpu]    = {xgb_gpu_peak_mb:.2f} MB")

    fold_preds_df = pd.DataFrame({
        "true_score": y_val, "pred_score": preds_val, "fold": fold_idx + 1,
    })

    result = {
        "fold"              : fold_idx + 1,
        "qwk"               : qwk_val,
        "mse"               : mse_val,
        "training_time_s"   : total_training_time,
        "inference_ms"      : inference_time_ms,
        "memory_mb_xgb_gpu" : xgb_gpu_peak_mb,
        "best_params"       : best_params,
    }
    return result, fold_preds_df, final_model


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    np.random.seed(CFG["seed"])

    print("\n[1/5] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])
    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())
    print(f"  Total sampel : {len(texts)}")

    print("\n[2/5] Mengekstrak fitur linguistik...")
    fe_list = []
    for i, text in enumerate(texts):
        if (i + 1) % 50 == 0 or i == 0: print(f"  [{i+1}/{len(texts)}] proses...")
        fe_list.append(extract_linguistic_features(text))
    features_raw = np.array([[d[k] for k in FEATURE_NAMES] for d in fe_list], dtype=np.float32)

    print(f"\n[3/5] Mengekstrak frozen embedding: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
    extractor = IndoBERTExtractor(CFG["model_name"])
    extractor.to(DEVICE).eval()

    embeddings, t_emb_total, gpu_mem_emb_mb = extract_embeddings(
        texts, tokenizer, extractor, CFG, DEVICE
    )
    emb_time_per_fold = t_emb_total / CFG["n_folds"]

    print(f"  Embedding shape        : {embeddings.shape}")
    print(f"  Waktu ekstraksi total  : {t_emb_total:.2f}s")
    print(f"  Waktu per fold (share) : {emb_time_per_fold:.2f}s")
    print(f"  Peak GPU memory (emb)  : {gpu_mem_emb_mb:.2f} MB")

    np.save(os.path.join(CFG["output_dir"], "embeddings_indobert_hyb.npy"), embeddings)

    X_all = np.concatenate([embeddings, features_raw], axis=1).astype(np.float32)
    y_all = scores.astype(np.float32)
    print(f"\n  Input matrix X_all: {X_all.shape}")

    print(f"\n[4/5] Memulai {CFG['n_folds']}-fold cross-validation...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

    all_results = []; all_preds_dfs = []; fold_models = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):

        train_idx, es_idx = inner_split(train_idx, y_all, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df, fold_model = run_fold(
                fold_idx, train_idx, val_idx,
                X_all, y_all, CFG, DEVICE,
                emb_time_per_fold=emb_time_per_fold, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
            fold_models.append(fold_model)
            model_path = os.path.join(CFG["output_dir"], f"xgb_model_fold{fold_idx+1}_indobert_hyb.json")
            fold_model.save_model(model_path)
        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1} gagal: {e}")
            traceback.print_exc()

    print(f"\n{'='*55}")
    print(f"  RINGKASAN HASIL — IndoBERT Hybrid")
    print(f"{'='*55}")

    results_df   = pd.DataFrame(all_results)
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)
    cols_print   = [c for c in results_df.columns if c != 'best_params']
    print(results_df[cols_print].to_string(index=False))

    print(f"\n  Rata-rata ± Std:")
    for metric in ["qwk", "mse", "training_time_s", "inference_ms", "memory_mb_xgb_gpu"]:
        m = results_df[metric].mean(); s = results_df[metric].std()
        print(f"    {metric:<28s}: {m:.4f} ± {s:.4f}")

    print(f"\n  Embedding extraction (global):")
    print(f"    peak_gpu_mem_emb_mb : {gpu_mem_emb_mb:.2f} MB")

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values, all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"],
    )
    overall_mse = mean_squared_error(all_preds_df["true_score"].values, all_preds_df["pred_score"].values)
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")

    experiment_data = {
        "config"                    : CFG,
        "fold_results"              : all_results,
        "fold_predictions"          : all_preds_df,
        "overall_qwk"               : overall_qwk,
        "overall_mse"               : overall_mse,
        "timestamp"                 : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "feature_names"             : FEATURE_NAMES,
        "embed_shape"               : embeddings.shape,
        "embedding_extraction_time_s" : t_emb_total,
        "embedding_time_per_fold_s" : emb_time_per_fold,
        "peak_gpu_mem_embedding_mb" : gpu_mem_emb_mb,
    }
    joblib_path = os.path.join(CFG["output_dir"], "experiment_results.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df[cols_print].to_csv(os.path.join(CFG["output_dir"], "cv_results_indobert_hyb.csv"), index=False)
    all_preds_df.to_csv(os.path.join(CFG["output_dir"], "all_predictions_indobert_hyb.csv"), index=False)

    print(f"\n  Hasil lengkap disimpan: {joblib_path}")
    return results_df


if __name__ == "__main__":
    main()

In [ ]:
"""
=============================================================
  AES — Paradigma B: DistilBERT-Indo Hybrid
  Metode   : Frozen embedding + Feature Engineering + XGBoost
  Dataset  : preprocessed_texts.pkl, scores.pkl
  Model    : cahya/distilbert-base-indonesian
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation (KFold)
  Referensi: Faseeh et al. (2024), Chen & Guestrin (2016)
=============================================================
"""

import os
import time
import threading
import traceback
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error
from nltk.tokenize import word_tokenize
from nlp_id.postag import PosTag
import xgboost as xgb
import pynvml


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    # ── data ──
    "essay"        : f"{ESSAY_PATH}preprocessed_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",

    # ── model transformer (frozen) ──
    "model_name"   : "cahya/distilbert-base-indonesian",
    "max_len"      : 256,
    "embed_dim"    : 768,
    "batch_size"   : 32,

    # ── XGBoost Static Hyperparameter ──
    "xgb_n_estimators"     : 300,
    "xgb_learning_rate"    : 0.05,
    "xgb_max_depth"        : 3,
    "xgb_subsample"        : 0.5,
    "xgb_colsample_bytree" : 0.5,
    "xgb_reg_lambda"       : 2.0,
    "xgb_reg_alpha"        : 0.1,
    "xgb_gamma"            : 0.1,
    "xgb_min_child_weight" : 20,
    "xgb_early_stop"       : 20,

    # ── cross-validation ──
    "n_folds"      : 10,
    "seed"         : 42,

    # ── output ──
    "output_dir"   : f"{OUTPUT_PATH}output_distilbert_hyb",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Model : {CFG['model_name']} (FROZEN)")


# ══════════════════════════════════════════════════════════
#  PYNVML — polling peak GPU untuk XGBoost (non-PyTorch)
# ══════════════════════════════════════════════════════════
def _gpu_poll_worker(stop_event, peak_holder, device_index=0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
    peak = 0
    while not stop_event.is_set():
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if info.used > peak:
            peak = info.used
        stop_event.wait(timeout=0.05)
    peak_holder.append(peak)
    pynvml.nvmlShutdown()


def measure_gpu_peak_mb(fn, device_index=0):
    stop_event  = threading.Event()
    peak_holder = []
    t = threading.Thread(target=_gpu_poll_worker,
                         args=(stop_event, peak_holder, device_index), daemon=True)
    t.start()
    result = fn()
    stop_event.set()
    t.join()
    peak_mb = peak_holder[0] / (1024 ** 2) if peak_holder else 0.0
    return result, peak_mb


# ══════════════════════════════════════════════════════════
#  LINGUISTIC FEATURE EXTRACTION
#
#  Fitur:
#  1. ttr              — lexical diversity (unique/total tokens)
#  2. word_count       — panjang respons, proxy cakupan konten
#  3. noun_count       — kepadatan konten faktual (NN, NNP)
#  4. verb_count       — elaborasi naratif/kausal (VB)
#  5. sentence_count   — kompleksitas sintaksis (proxy)
#  6. avg_word_length  — kompleksitas morfologi (proxy, Indonesian-specific)
#
#  Library:
#  - nltk.word_tokenize  : tokenisasi kata
#  - nltk.sent_tokenize  : deteksi batas kalimat
#  - nlp_id.postag       : POS tagging bahasa Indonesia
# ══════════════════════════════════════════════════════════
postag = PosTag()

POS_NOUN = {'NN', 'NNP'}
POS_VERB = {'VB'}

FEATURE_NAMES = [
    'ttr',
    'word_count',
    'noun_count',
    'verb_count',
    'sentence_count',
    'avg_word_length',
]


def extract_linguistic_features(text: str) -> dict:
    """
    Ekstraksi 6 fitur linguistik dari teks yang sudah dipreprocess
    (dengan tanda baca bermakna dipertahankan).

    Args:
        text : teks preprocessed (output preprocess_text)

    Returns:
        dict dengan 6 fitur (urutan sesuai FEATURE_NAMES)
    """
    if not text:
        return {f: 0.0 for f in FEATURE_NAMES}

    # --- Tokenisasi kata ---
    words = word_tokenize(text)
    # Filter: hanya kata alfanumerik (bukan tanda baca yang ikut ter-tokenize)
    word_tokens = [w for w in words if w.isalnum()]
    n_words = len(word_tokens)

    if n_words == 0:
        return {f: 0.0 for f in FEATURE_NAMES}

    # 1. TTR — type-token ratio
    ttr = len(set(word_tokens)) / n_words

    # 2. Word count
    word_count = float(n_words)

    # 3 & 4. POS counts (noun, verb) via nlp_id
    pos_tags   = postag.get_pos_tag(text)
    noun_count = sum(1 for _, pos in pos_tags if pos in POS_NOUN)
    verb_count = sum(1 for _, pos in pos_tags if pos in POS_VERB)

    # 5. Sentence count — valid karena tanda baca dipertahankan
    sentences      = sent_tokenize(text)
    sentence_count = float(max(len(sentences), 1))

    # 6. Average word length (characters per token)
    #    Proxy morphological complexity untuk bahasa Indonesia aglutinatif
    avg_word_length = sum(len(w) for w in word_tokens) / n_words

    return {
        'ttr'            : ttr,
        'word_count'     : word_count,
        'noun_count'     : float(noun_count),
        'verb_count'     : float(verb_count),
        'sentence_count' : sentence_count,
        'avg_word_length': avg_word_length,
    }


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating=None, max_rating=None):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)

    if min_rating is None:
        min_rating = min(y_true.min(), y_pred.min())
    if max_rating is None:
        max_rating = max(y_true.max(), y_pred.max())

    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1

    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored

    weights = np.zeros((num_ratings, num_ratings))
    for i in range(num_ratings):
        for j in range(num_ratings):
            weights[i][j] = ((i - j) ** 2) / ((num_ratings - 1) ** 2)

    numerator   = np.sum(weights * conf_mat)
    denominator = np.sum(weights * expected)
    if denominator == 0:
        return 0.0
    return 1.0 - numerator / denominator


# ══════════════════════════════════════════════════════════
#  DATASET — untuk ekstraksi embedding saja
# ══════════════════════════════════════════════════════════
class EssayEmbedDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


# ══════════════════════════════════════════════════════════
#  FROZEN EXTRACTOR
# ══════════════════════════════════════════════════════════
class DistilBERTExtractor(nn.Module):
    """
    Frozen embedding extractor.
    Bobot tidak pernah diperbarui — hanya untuk ekstraksi representasi teks.
    Representasi via [CLS] token, output dim 768.
    """
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        emb = out.last_hidden_state[:, 0, :]   # [CLS] token, dim 768
        return emb


def extract_embeddings(texts, tokenizer, model, cfg, device):
    """
    Ekstrak embedding dari seluruh dataset dalam satu pass.
    GPU memory diukur dengan torch.cuda.max_memory_allocated().
    Waktu diukur dengan time.perf_counter().

    Returns:
        embeddings      : np.ndarray (N, embed_dim)
        t_emb_total     : float, waktu ekstraksi dalam detik
        gpu_mem_emb_mb  : float, peak GPU memory saat ekstraksi dalam MB
    """
    model.eval()
    dataset  = EssayEmbedDataset(texts, tokenizer, cfg["max_len"])
    loader   = DataLoader(dataset, batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)
    all_embs = []

    # Reset GPU memory counter sebelum ekstraksi
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    t_emb_start = time.perf_counter()

    with torch.no_grad():
        for batch in loader:
            embs = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
            )
            all_embs.append(embs.cpu().numpy())

    t_emb_total = time.perf_counter() - t_emb_start

    # Peak GPU memory saat ekstraksi embedding
    if device.type == "cuda":
        gpu_mem_emb_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    else:
        gpu_mem_emb_mb = 0.0

    return np.vstack(all_embs), t_emb_total, gpu_mem_emb_mb


# ══════════════════════════════════════════════════════════
#  XGBOOST — STATIC HYPERPARAMETER
# ══════════════════════════════════════════════════════════
def train_xgboost_static(X_train, y_train, X_val, y_val, cfg):
    params = {
        "n_estimators"    : cfg["xgb_n_estimators"],
        "learning_rate"   : cfg["xgb_learning_rate"],
        "max_depth"       : cfg["xgb_max_depth"],
        "subsample"       : cfg["xgb_subsample"],
        "colsample_bytree": cfg["xgb_colsample_bytree"],
        "booster"         : "gbtree",
        "reg_lambda"      : cfg["xgb_reg_lambda"],
        "reg_alpha"       : cfg["xgb_reg_alpha"],
        "gamma"           : cfg["xgb_gamma"],
        "min_child_weight": cfg["xgb_min_child_weight"],
        "tree_method"     : "hist",
        "device"          : "cuda",
        "random_state"    : cfg["seed"],
        "verbosity"       : 0,
    }

    model = xgb.XGBRegressor(
        **params,
        early_stopping_rounds=cfg["xgb_early_stop"]
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    return model, params


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx,
             X_all, y_all, cfg, device,
             emb_time_per_fold, es_idx=None):
    """
    Parameters
    ----------
    emb_time_per_fold : float
        t_emb_total / n_folds — porsi embedding extraction time untuk fold ini.
    """
    print(f"\n{'='*55}")
    print(f"  FOLD {fold_idx + 1}/{cfg['n_folds']}")
    print(f"{'='*55}")

    X_train = X_all[train_idx]
    X_val   = X_all[val_idx]
    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    X_es, y_es = X_all[es_idx], y_all[es_idx]   # [FIX-LEAKAGE] early-stopping set

    print(f"  Train: {len(X_train)} | Val: {len(X_val)}")
    print(f"  Input dim: {X_train.shape[1]} ({cfg['embed_dim']} embed + {len(FEATURE_NAMES)} FE)")
    print("  Training XGBoost dengan hyperparameter statis...")

    gpu_index = device.index if (device.type == "cuda" and device.index is not None) else 0

    t_xgb_start = time.perf_counter()
    (final_model, best_params), xgb_gpu_peak_mb = measure_gpu_peak_mb(
        lambda: train_xgboost_static(X_train, y_train, X_es, y_es, cfg),
        device_index=gpu_index,
    )
    t_xgb_train = time.perf_counter() - t_xgb_start

    total_training_time = emb_time_per_fold + t_xgb_train

    # ── Inference time per sampel (warm-up + pengukuran) ──
    _ = final_model.predict(X_val[:1])   # warm-up
    t_infer_start = time.perf_counter()
    _ = final_model.predict(X_val[:1])
    inference_time_ms = (time.perf_counter() - t_infer_start) * 1000

    # ── Prediksi seluruh validation set untuk metrik ──
    preds_val = final_model.predict(X_val)
    mse_val   = mean_squared_error(y_val, preds_val)
    qwk_val   = quadratic_weighted_kappa(y_val, preds_val, cfg["score_min"], cfg["score_max"])

    print(f"\n  → Fold {fold_idx+1} FINAL:")
    print(f"    QWK                    = {qwk_val:.4f}")
    print(f"    MSE                    = {mse_val:.4f}")
    print(f"    training_time_s        = {total_training_time:.2f}s "
          f"(emb_share={emb_time_per_fold:.2f}s + xgb={t_xgb_train:.2f}s)")
    print(f"    inference_ms           = {inference_time_ms:.4f} ms/sampel")
    print(f"    memory_mb [xgb_gpu]    = {xgb_gpu_peak_mb:.2f} MB")

    fold_preds_df = pd.DataFrame({
        "true_score": y_val,
        "pred_score": preds_val,
        "fold"      : fold_idx + 1,
    })

    result = {
        "fold"              : fold_idx + 1,
        "qwk"               : qwk_val,
        "mse"               : mse_val,
        "training_time_s"   : total_training_time,
        "inference_ms"      : inference_time_ms,
        "memory_mb_xgb_gpu" : xgb_gpu_peak_mb,
        "best_params"       : best_params,
    }
    return result, fold_preds_df, final_model


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    np.random.seed(CFG["seed"])

    # ── 1. Load data ──────────────────────────────────────
    print("\n[1/5] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])

    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())

    print(f"  Total sampel : {len(texts)}")
    print(f"  Rentang skor : {CFG['score_min']} – {CFG['score_max']}")
    print(f"  Rata-rata    : {scores.mean():.2f} ± {scores.std():.2f}")

    # ── 2. Feature Engineering ────────────────────────────
    print("\n[2/5] Mengekstrak fitur linguistik...")
    fe_list = []
    for i, text in enumerate(texts):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  [{i+1}/{len(texts)}] proses...")
        fe_list.append(extract_linguistic_features(text))
    features_raw = np.array(
        [[d[k] for k in FEATURE_NAMES] for d in fe_list],
        dtype=np.float32,
    )
    print(f"  Dimensi fitur : {features_raw.shape[1]} — {FEATURE_NAMES}")

    # ── 3. Ekstrak embedding (SATU KALI, frozen) ──────────
    print(f"\n[3/5] Mengekstrak frozen embedding: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
    extractor = DistilBERTExtractor(CFG["model_name"])
    extractor.to(DEVICE)
    extractor.eval()

    embeddings, t_emb_total, gpu_mem_emb_mb = extract_embeddings(
        texts, tokenizer, extractor, CFG, DEVICE
    )

    # Porsi embedding time per fold
    emb_time_per_fold = t_emb_total / CFG["n_folds"]

    print(f"  Embedding shape        : {embeddings.shape}")
    print(f"  Waktu ekstraksi total  : {t_emb_total:.2f}s")
    print(f"  Waktu per fold (share) : {emb_time_per_fold:.2f}s")
    print(f"  Peak GPU memory (emb)  : {gpu_mem_emb_mb:.2f} MB")

    # Simpan embedding untuk reproducibility
    emb_path = os.path.join(CFG["output_dir"], "embeddings_distilbert_hyb.npy")
    np.save(emb_path, embeddings)
    print(f"  Embedding disimpan: {emb_path}")

    # ── 4. Gabungkan embedding + FE ───────────────────────
    X_all = np.concatenate([embeddings, features_raw], axis=1).astype(np.float32)
    y_all = scores.astype(np.float32)
    print(f"\n  Input matrix X_all: {X_all.shape}")
    print(f"  ({CFG['embed_dim']} embedding dim + {len(FEATURE_NAMES)} FE dim)")

    # ── 5. Cross-Validation ───────────────────────────────
    print(f"\n[4/5] Memulai {CFG['n_folds']}-fold cross-validation...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

    all_results   = []
    all_preds_dfs = []
    fold_models   = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):

        train_idx, es_idx = inner_split(train_idx, y_all, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df, fold_model = run_fold(
                fold_idx, train_idx, val_idx,
                X_all, y_all, CFG, DEVICE,
                emb_time_per_fold=emb_time_per_fold, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
            fold_models.append(fold_model)

            model_path = os.path.join(
                CFG["output_dir"],
                f"xgb_model_fold{fold_idx+1}_distilbert_hyb.json"
            )
            fold_model.save_model(model_path)
            print(f"  Model fold {fold_idx+1} disimpan: {model_path}")

        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1} gagal: {e}")
            traceback.print_exc()

    # ── 6. Ringkasan & Simpan ─────────────────────────────
    print(f"\n{'='*55}")
    print(f"  RINGKASAN HASIL — DistilBERT-Indo Hybrid")
    print(f"{'='*55}")

    results_df   = pd.DataFrame(all_results)
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)

    cols_print = [c for c in results_df.columns if c != 'best_params']
    print(results_df[cols_print].to_string(index=False))

    print(f"\n  Rata-rata ± Std:")
    for metric in ["qwk", "mse", "training_time_s", "inference_ms", "memory_mb_xgb_gpu"]:
        m = results_df[metric].mean()
        s = results_df[metric].std()
        print(f"    {metric:<28s}: {m:.4f} ± {s:.4f}")

    print(f"\n  Embedding extraction (global):")
    print(f"    peak_gpu_mem_emb_mb : {gpu_mem_emb_mb:.2f} MB")

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"],
    )
    overall_mse = mean_squared_error(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values,
    )
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")

    experiment_data = {
        "config"                       : CFG,
        "fold_results"                 : all_results,
        "fold_predictions"             : all_preds_df,
        "overall_qwk"                  : overall_qwk,
        "overall_mse"                  : overall_mse,
        "timestamp"                    : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "feature_names"                : FEATURE_NAMES,
        "embed_shape"                  : embeddings.shape,
        "embedding_extraction_time_s"  : t_emb_total,
        "embedding_time_per_fold_s"    : emb_time_per_fold,
        "peak_gpu_mem_embedding_mb"    : gpu_mem_emb_mb,
    }
    joblib_path = os.path.join(CFG["output_dir"], "experiment_results.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df[cols_print].to_csv(
        os.path.join(CFG["output_dir"], "cv_results_distilbert_hyb.csv"),
        index=False,
    )
    all_preds_df.to_csv(
        os.path.join(CFG["output_dir"], "all_predictions_distilbert_hyb.csv"),
        index=False,
    )

    print(f"\n  Hasil lengkap disimpan: {joblib_path}")
    print(f"  Output direktori      : {CFG['output_dir']}/")

    return results_df


if __name__ == "__main__":
    main()

---


# ABLASI ML- NO FA

In [ ]:
"""
=============================================================
  AES — Paradigma B: SBERT Hybrid (tanpa fitur linguistik)
  Metode   : Frozen embedding + XGBoost  (TANPA Feature Engineering)
  Dataset  : preprocessed_sbert_texts.pkl, scores.pkl  (SAMA dengan versi +FE)
  Model    : denaya/indoSBERT-large
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation (KFold)
  Referensi: Iqbal et al. (2024), Chen & Guestrin (2016)
=============================================================
CATATAN ABLASI:
  - Satu-satunya perbedaan dengan cell "SBERT Hybrid": X_all
    HANYA berisi embedding frozen (256 dim), TIDAK digabung
    dengan 6 fitur linguistik.
  - Dataset, n_folds, seed, hyperparameter XGBoost dibuat identik
    dengan versi +FE agar selisih QWK/MSE dapat diatribusikan
    pada fitur linguistik, bukan pada confound lain.
=============================================================
"""

import os
import time
import threading
import traceback
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import pynvml


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay"        : f"{ESSAY_PATH}preprocessed_sbert_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",
    "model_name"   : "denaya/indoSBERT-large",
    "max_len"      : 256,
    "embed_dim"    : 256,
    "batch_size"   : 32,
    "xgb_n_estimators"     : 300,
    "xgb_learning_rate"    : 0.05,
    "xgb_max_depth"        : 3,
    "xgb_subsample"        : 0.5,
    "xgb_colsample_bytree" : 0.5,
    "xgb_reg_lambda"       : 2.0,
    "xgb_reg_alpha"        : 0.1,
    "xgb_gamma"            : 0.1,
    "xgb_min_child_weight" : 20,
    "xgb_early_stop"       : 20,
    "n_folds"      : 10,
    "seed"         : 42,
    "output_dir"   : f"{OUTPUT_PATH}output_sbert_hyb_no_fe",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {DEVICE}")
print(f"Model          : {CFG['model_name']} (FROZEN)")
print(f"Fitur tambahan : TIDAK DIGUNAKAN (embedding only)")


# ══════════════════════════════════════════════════════════
#  PYNVML — polling peak GPU untuk XGBoost (non-PyTorch)
# ══════════════════════════════════════════════════════════
def _gpu_poll_worker(stop_event, peak_holder, device_index=0):
    """Thread worker: poll VRAM setiap 50ms, simpan nilai tertinggi."""
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
    peak = 0
    while not stop_event.is_set():
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if info.used > peak:
            peak = info.used
        stop_event.wait(timeout=0.05)
    peak_holder.append(peak)
    pynvml.nvmlShutdown()


def measure_gpu_peak_mb(fn, device_index=0):
    """
    Jalankan fn() sambil polling VRAM via pynvml.
    Return: (result_of_fn, peak_mb)
    """
    stop_event  = threading.Event()
    peak_holder = []
    t = threading.Thread(
        target=_gpu_poll_worker,
        args=(stop_event, peak_holder, device_index),
        daemon=True,
    )
    t.start()
    result = fn()
    stop_event.set()
    t.join()
    peak_mb = peak_holder[0] / (1024 ** 2) if peak_holder else 0.0
    return result, peak_mb


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating=None, max_rating=None):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)
    if min_rating is None: min_rating = min(y_true.min(), y_pred.min())
    if max_rating is None: max_rating = max(y_true.max(), y_pred.max())
    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1
    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored
    weights    = np.zeros((num_ratings, num_ratings))
    for i in range(num_ratings):
        for j in range(num_ratings):
            weights[i][j] = ((i - j) ** 2) / ((num_ratings - 1) ** 2)
    numerator   = np.sum(weights * conf_mat)
    denominator = np.sum(weights * expected)
    return 0.0 if denominator == 0 else 1.0 - numerator / denominator


# ══════════════════════════════════════════════════════════
#  DATASET — untuk ekstraksi embedding saja
# ══════════════════════════════════════════════════════════
class EssayEmbedDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}


# ══════════════════════════════════════════════════════════
#  FROZEN EXTRACTOR
# ══════════════════════════════════════════════════════════
class SBERTExtractor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out      = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_emb = out.last_hidden_state
        mask_exp  = attention_mask.unsqueeze(-1).float()
        sum_emb   = torch.sum(token_emb * mask_exp, dim=1)
        sum_mask  = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
        return sum_emb / sum_mask


def extract_embeddings(texts, tokenizer, model, cfg, device):
    """
    Ekstrak embedding + ukur peak GPU via torch.
    Returns: embeddings, t_emb_total, gpu_mem_emb_mb
    """
    model.eval()
    dataset  = EssayEmbedDataset(texts, tokenizer, cfg["max_len"])
    loader   = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
    all_embs = []

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    t_start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            embs = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            all_embs.append(embs.cpu().numpy())
    t_total = time.perf_counter() - t_start

    gpu_mem_mb = (torch.cuda.max_memory_allocated(device) / (1024 ** 2)
                  if device.type == "cuda" else 0.0)

    return np.vstack(all_embs), t_total, gpu_mem_mb


# ══════════════════════════════════════════════════════════
#  XGBOOST
# ══════════════════════════════════════════════════════════
def train_xgboost_static(X_train, y_train, X_val, y_val, cfg):
    params = {
        "n_estimators"    : cfg["xgb_n_estimators"],
        "learning_rate"   : cfg["xgb_learning_rate"],
        "max_depth"       : cfg["xgb_max_depth"],
        "subsample"       : cfg["xgb_subsample"],
        "colsample_bytree": cfg["xgb_colsample_bytree"],
        "booster"         : "gbtree",
        "reg_lambda"      : cfg["xgb_reg_lambda"],
        "reg_alpha"       : cfg["xgb_reg_alpha"],
        "gamma"           : cfg["xgb_gamma"],
        "min_child_weight": cfg["xgb_min_child_weight"],
        "tree_method"     : "hist",
        "device"          : "cuda",
        "random_state"    : cfg["seed"],
        "verbosity"       : 0,
    }
    model = xgb.XGBRegressor(**params, early_stopping_rounds=cfg["xgb_early_stop"])
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model, params


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx, X_all, y_all, cfg, device, emb_time_per_fold, es_idx=None):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold_idx + 1}/{cfg['n_folds']}")
    print(f"{'='*55}")

    X_train = X_all[train_idx]; X_val = X_all[val_idx]
    y_train = y_all[train_idx]; y_val = y_all[val_idx]
    X_es, y_es = X_all[es_idx], y_all[es_idx]   # [FIX-LEAKAGE] early-stopping set

    print(f"  Train: {len(X_train)} | Val: {len(X_val)}")

    gpu_index = device.index if (device.type == "cuda" and device.index is not None) else 0

    t_xgb_start = time.perf_counter()
    (final_model, best_params), xgb_gpu_peak_mb = measure_gpu_peak_mb(
        lambda: train_xgboost_static(X_train, y_train, X_es, y_es, cfg),
        device_index=gpu_index,
    )
    t_xgb_train = time.perf_counter() - t_xgb_start

    total_training_time = emb_time_per_fold + t_xgb_train

    _ = final_model.predict(X_val[:1])   # warm-up
    t_infer_start = time.perf_counter()
    _ = final_model.predict(X_val[:1])
    inference_time_ms = (time.perf_counter() - t_infer_start) * 1000

    preds_val = final_model.predict(X_val)
    mse_val   = mean_squared_error(y_val, preds_val)
    qwk_val   = quadratic_weighted_kappa(y_val, preds_val, cfg["score_min"], cfg["score_max"])

    print(f"\n  → Fold {fold_idx+1} FINAL:")
    print(f"    QWK                    = {qwk_val:.4f}")
    print(f"    MSE                    = {mse_val:.4f}")
    print(f"    training_time_s        = {total_training_time:.2f}s "
          f"(emb_share={emb_time_per_fold:.2f}s + xgb={t_xgb_train:.2f}s)")
    print(f"    inference_ms           = {inference_time_ms:.4f} ms/sampel")
    print(f"    memory_mb [xgb_gpu]    = {xgb_gpu_peak_mb:.2f} MB")

    fold_preds_df = pd.DataFrame({
        "true_score": y_val, "pred_score": preds_val, "fold": fold_idx + 1,
    })

    result = {
        "fold"              : fold_idx + 1,
        "qwk"               : qwk_val,
        "mse"               : mse_val,
        "training_time_s"   : total_training_time,
        "inference_ms"      : inference_time_ms,
        "memory_mb_xgb_gpu" : xgb_gpu_peak_mb,
        "best_params"       : best_params,
    }
    return result, fold_preds_df, final_model


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    np.random.seed(CFG["seed"])

    print("\n[1/4] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])
    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())
    print(f"  Total sampel : {len(texts)}")
    print(f"  Rentang skor : {CFG['score_min']} – {CFG['score_max']}")

    print(f"\n[2/4] Mengekstrak frozen embedding: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
    extractor = SBERTExtractor(CFG["model_name"])
    extractor.to(DEVICE).eval()

    embeddings, t_emb_total, gpu_mem_emb_mb = extract_embeddings(
        texts, tokenizer, extractor, CFG, DEVICE
    )
    emb_time_per_fold = t_emb_total / CFG["n_folds"]

    print(f"  Embedding shape        : {embeddings.shape}")
    print(f"  Waktu ekstraksi total  : {t_emb_total:.2f}s")
    print(f"  Waktu per fold (share) : {emb_time_per_fold:.2f}s")
    print(f"  Peak GPU memory (emb)  : {gpu_mem_emb_mb:.2f} MB")

    emb_path = os.path.join(CFG["output_dir"], "embeddings_sbert_hyb_no_fe.npy")
    np.save(emb_path, embeddings)

    X_all = embeddings.astype(np.float32)   # TANPA fitur linguistik
    y_all = scores.astype(np.float32)
    print(f"\n  Input matrix X_all: {X_all.shape}  (embedding only, tanpa FE)")

    print(f"\n[3/4] Memulai {CFG['n_folds']}-fold cross-validation...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

    all_results = []; all_preds_dfs = []; fold_models = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):

        train_idx, es_idx = inner_split(train_idx, y_all, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df, fold_model = run_fold(
                fold_idx, train_idx, val_idx,
                X_all, y_all, CFG, DEVICE,
                emb_time_per_fold=emb_time_per_fold, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
            fold_models.append(fold_model)
            model_path = os.path.join(CFG["output_dir"], f"xgb_model_fold{fold_idx+1}_sbert_hyb_no_fe.json")
            fold_model.save_model(model_path)
        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1} gagal: {e}")
            traceback.print_exc()

    print(f"\n[4/4] Menyimpan hasil...")
    print(f"\n{'='*55}")
    print(f"  RINGKASAN HASIL — SBERT Hybrid (tanpa fitur linguistik)")
    print(f"{'='*55}")

    results_df   = pd.DataFrame(all_results)
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)

    cols_print = [c for c in results_df.columns if c != 'best_params']
    print(results_df[cols_print].to_string(index=False))

    print(f"\n  Rata-rata ± Std:")
    for metric in ["qwk", "mse", "training_time_s", "inference_ms", "memory_mb_xgb_gpu"]:
        m = results_df[metric].mean(); s = results_df[metric].std()
        print(f"    {metric:<28s}: {m:.4f} ± {s:.4f}")

    print(f"\n  Embedding extraction (global):")
    print(f"    peak_gpu_mem_emb_mb : {gpu_mem_emb_mb:.2f} MB")

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values, all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"],
    )
    overall_mse = mean_squared_error(all_preds_df["true_score"].values, all_preds_df["pred_score"].values)
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")

    experiment_data = {
        "config"                    : CFG,
        "fold_results"              : all_results,
        "fold_predictions"          : all_preds_df,
        "overall_qwk"               : overall_qwk,
        "overall_mse"               : overall_mse,
        "timestamp"                 : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "feature_names"             : None,
        "embed_shape"               : embeddings.shape,
        "embedding_extraction_time_s" : t_emb_total,
        "embedding_time_per_fold_s" : emb_time_per_fold,
        "peak_gpu_mem_embedding_mb" : gpu_mem_emb_mb,
        "paradigm"                  : "hybrid (frozen embedding only, no linguistic features)",
    }
    joblib_path = os.path.join(CFG["output_dir"], "experiment_results_no_fe.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df[cols_print].to_csv(os.path.join(CFG["output_dir"], "cv_results_sbert_hyb_no_fe.csv"), index=False)
    all_preds_df.to_csv(os.path.join(CFG["output_dir"], "all_predictions_sbert_hyb_no_fe.csv"), index=False)

    print(f"\n  Hasil lengkap disimpan: {joblib_path}")
    return results_df


if __name__ == "__main__":
    main()


---


In [ ]:
"""
=============================================================
  AES — Paradigma B: IndoBERT Hybrid (tanpa fitur linguistik)
  Metode   : Frozen embedding + XGBoost  (TANPA Feature Engineering)
  Dataset  : preprocessed_texts.pkl, scores.pkl  (SAMA dengan versi +FE)
  Model    : indobenchmark/indobert-base-p2
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation (KFold)
  Referensi: Faseeh et al. (2024), Chen & Guestrin (2016)
=============================================================
CATATAN ABLASI:
  - Satu-satunya perbedaan dengan cell "IndoBERT Hybrid": X_all
    HANYA berisi embedding frozen ([CLS], 768 dim), TIDAK
    digabung dengan 6 fitur linguistik.
  - Dataset, n_folds, seed, hyperparameter XGBoost dibuat identik
    dengan versi +FE agar selisih QWK/MSE dapat diatribusikan
    pada fitur linguistik, bukan pada confound lain.
=============================================================
"""

import os
import time
import threading
import traceback
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import pynvml


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay"        : f"{ESSAY_PATH}preprocessed_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",
    "model_name"   : "indobenchmark/indobert-base-p2",
    "max_len"      : 256,
    "embed_dim"    : 768,
    "batch_size"   : 64,
    "xgb_n_estimators"     : 300,
    "xgb_learning_rate"    : 0.05,
    "xgb_max_depth"        : 3,
    "xgb_subsample"        : 0.5,
    "xgb_colsample_bytree" : 0.5,
    "xgb_reg_lambda"       : 2.0,
    "xgb_reg_alpha"        : 0.1,
    "xgb_gamma"            : 0.1,
    "xgb_min_child_weight" : 20,
    "xgb_early_stop"       : 20,
    "n_folds"      : 10,
    "seed"         : 42,
    "output_dir"   : f"{OUTPUT_PATH}Hasil eksperiment/output_indobert_hyb_no_fe",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {DEVICE}")
print(f"Model          : {CFG['model_name']} (FROZEN)")
print(f"Fitur tambahan : TIDAK DIGUNAKAN (embedding only)")


# ══════════════════════════════════════════════════════════
#  PYNVML — polling peak GPU untuk XGBoost (non-PyTorch)
# ══════════════════════════════════════════════════════════
def _gpu_poll_worker(stop_event, peak_holder, device_index=0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
    peak = 0
    while not stop_event.is_set():
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if info.used > peak:
            peak = info.used
        stop_event.wait(timeout=0.05)
    peak_holder.append(peak)
    pynvml.nvmlShutdown()


def measure_gpu_peak_mb(fn, device_index=0):
    stop_event  = threading.Event()
    peak_holder = []
    t = threading.Thread(target=_gpu_poll_worker,
                         args=(stop_event, peak_holder, device_index), daemon=True)
    t.start()
    result = fn()
    stop_event.set()
    t.join()
    peak_mb = peak_holder[0] / (1024 ** 2) if peak_holder else 0.0
    return result, peak_mb


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating=None, max_rating=None):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)
    if min_rating is None: min_rating = min(y_true.min(), y_pred.min())
    if max_rating is None: max_rating = max(y_true.max(), y_pred.max())
    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1
    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored
    weights    = np.zeros((num_ratings, num_ratings))
    for i in range(num_ratings):
        for j in range(num_ratings):
            weights[i][j] = ((i - j) ** 2) / ((num_ratings - 1) ** 2)
    numerator   = np.sum(weights * conf_mat)
    denominator = np.sum(weights * expected)
    return 0.0 if denominator == 0 else 1.0 - numerator / denominator


# ══════════════════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════════════════
class EssayEmbedDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}


# ══════════════════════════════════════════════════════════
#  FROZEN EXTRACTOR — [CLS] token
# ══════════════════════════════════════════════════════════
class IndoBERTExtractor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]   # [CLS] dim 768


def extract_embeddings(texts, tokenizer, model, cfg, device):
    model.eval()
    dataset  = EssayEmbedDataset(texts, tokenizer, cfg["max_len"])
    loader   = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
    all_embs = []

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    t_start = time.perf_counter()
    with torch.no_grad():
        for batch in loader:
            embs = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            all_embs.append(embs.cpu().numpy())
    t_total = time.perf_counter() - t_start

    gpu_mem_mb = (torch.cuda.max_memory_allocated(device) / (1024 ** 2)
                  if device.type == "cuda" else 0.0)

    return np.vstack(all_embs), t_total, gpu_mem_mb


# ══════════════════════════════════════════════════════════
#  XGBOOST
# ══════════════════════════════════════════════════════════
def train_xgboost_static(X_train, y_train, X_val, y_val, cfg):
    params = {
        "n_estimators"    : cfg["xgb_n_estimators"],
        "learning_rate"   : cfg["xgb_learning_rate"],
        "max_depth"       : cfg["xgb_max_depth"],
        "subsample"       : cfg["xgb_subsample"],
        "colsample_bytree": cfg["xgb_colsample_bytree"],
        "booster"         : "gbtree",
        "reg_lambda"      : cfg["xgb_reg_lambda"],
        "reg_alpha"       : cfg["xgb_reg_alpha"],
        "gamma"           : cfg["xgb_gamma"],
        "min_child_weight": cfg["xgb_min_child_weight"],
        "tree_method"     : "hist",
        "device"          : "cuda",
        "random_state"    : cfg["seed"],
        "verbosity"       : 0,
    }
    model = xgb.XGBRegressor(**params, early_stopping_rounds=cfg["xgb_early_stop"])
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model, params


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx, X_all, y_all, cfg, device, emb_time_per_fold, es_idx=None):
    print(f"\n{'='*55}")
    print(f"  FOLD {fold_idx + 1}/{cfg['n_folds']}")
    print(f"{'='*55}")

    X_train = X_all[train_idx]; X_val = X_all[val_idx]
    y_train = y_all[train_idx]; y_val = y_all[val_idx]
    X_es, y_es = X_all[es_idx], y_all[es_idx]   # [FIX-LEAKAGE] early-stopping set
    print(f"  Train: {len(X_train)} | Val: {len(X_val)}")

    gpu_index = device.index if (device.type == "cuda" and device.index is not None) else 0

    t_xgb_start = time.perf_counter()
    (final_model, best_params), xgb_gpu_peak_mb = measure_gpu_peak_mb(
        lambda: train_xgboost_static(X_train, y_train, X_es, y_es, cfg),
        device_index=gpu_index,
    )
    t_xgb_train = time.perf_counter() - t_xgb_start

    total_training_time = emb_time_per_fold + t_xgb_train

    _ = final_model.predict(X_val[:1])   # warm-up
    t_infer_start = time.perf_counter()
    _ = final_model.predict(X_val[:1])
    inference_time_ms = (time.perf_counter() - t_infer_start) * 1000

    preds_val = final_model.predict(X_val)
    mse_val   = mean_squared_error(y_val, preds_val)
    qwk_val   = quadratic_weighted_kappa(y_val, preds_val, cfg["score_min"], cfg["score_max"])

    print(f"\n  → Fold {fold_idx+1} FINAL:")
    print(f"    QWK                    = {qwk_val:.4f}")
    print(f"    MSE                    = {mse_val:.4f}")
    print(f"    training_time_s        = {total_training_time:.2f}s "
          f"(emb_share={emb_time_per_fold:.2f}s + xgb={t_xgb_train:.2f}s)")
    print(f"    inference_ms           = {inference_time_ms:.4f} ms/sampel")
    print(f"    memory_mb [xgb_gpu]    = {xgb_gpu_peak_mb:.2f} MB")

    fold_preds_df = pd.DataFrame({
        "true_score": y_val, "pred_score": preds_val, "fold": fold_idx + 1,
    })

    result = {
        "fold"              : fold_idx + 1,
        "qwk"               : qwk_val,
        "mse"               : mse_val,
        "training_time_s"   : total_training_time,
        "inference_ms"      : inference_time_ms,
        "memory_mb_xgb_gpu" : xgb_gpu_peak_mb,
        "best_params"       : best_params,
    }
    return result, fold_preds_df, final_model


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    np.random.seed(CFG["seed"])

    print("\n[1/4] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])
    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())
    print(f"  Total sampel : {len(texts)}")

    print(f"\n[2/4] Mengekstrak frozen embedding: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
    extractor = IndoBERTExtractor(CFG["model_name"])
    extractor.to(DEVICE).eval()

    embeddings, t_emb_total, gpu_mem_emb_mb = extract_embeddings(
        texts, tokenizer, extractor, CFG, DEVICE
    )
    emb_time_per_fold = t_emb_total / CFG["n_folds"]

    print(f"  Embedding shape        : {embeddings.shape}")
    print(f"  Waktu ekstraksi total  : {t_emb_total:.2f}s")
    print(f"  Waktu per fold (share) : {emb_time_per_fold:.2f}s")
    print(f"  Peak GPU memory (emb)  : {gpu_mem_emb_mb:.2f} MB")

    np.save(os.path.join(CFG["output_dir"], "embeddings_indobert_hyb_no_fe.npy"), embeddings)

    X_all = embeddings.astype(np.float32)   # TANPA fitur linguistik
    y_all = scores.astype(np.float32)
    print(f"\n  Input matrix X_all: {X_all.shape}  (embedding only, tanpa FE)")

    print(f"\n[3/4] Memulai {CFG['n_folds']}-fold cross-validation...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

    all_results = []; all_preds_dfs = []; fold_models = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):

        train_idx, es_idx = inner_split(train_idx, y_all, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df, fold_model = run_fold(
                fold_idx, train_idx, val_idx,
                X_all, y_all, CFG, DEVICE,
                emb_time_per_fold=emb_time_per_fold, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
            fold_models.append(fold_model)
            model_path = os.path.join(CFG["output_dir"], f"xgb_model_fold{fold_idx+1}_indobert_hyb_no_fe.json")
            fold_model.save_model(model_path)
        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1} gagal: {e}")
            traceback.print_exc()

    print(f"\n[4/4] Menyimpan hasil...")
    print(f"\n{'='*55}")
    print(f"  RINGKASAN HASIL — IndoBERT Hybrid (tanpa fitur linguistik)")
    print(f"{'='*55}")

    results_df   = pd.DataFrame(all_results)
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)
    cols_print   = [c for c in results_df.columns if c != 'best_params']
    print(results_df[cols_print].to_string(index=False))

    print(f"\n  Rata-rata ± Std:")
    for metric in ["qwk", "mse", "training_time_s", "inference_ms", "memory_mb_xgb_gpu"]:
        m = results_df[metric].mean(); s = results_df[metric].std()
        print(f"    {metric:<28s}: {m:.4f} ± {s:.4f}")

    print(f"\n  Embedding extraction (global):")
    print(f"    peak_gpu_mem_emb_mb : {gpu_mem_emb_mb:.2f} MB")

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values, all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"],
    )
    overall_mse = mean_squared_error(all_preds_df["true_score"].values, all_preds_df["pred_score"].values)
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")

    experiment_data = {
        "config"                    : CFG,
        "fold_results"              : all_results,
        "fold_predictions"          : all_preds_df,
        "overall_qwk"               : overall_qwk,
        "overall_mse"               : overall_mse,
        "timestamp"                 : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "feature_names"             : None,
        "embed_shape"               : embeddings.shape,
        "embedding_extraction_time_s" : t_emb_total,
        "embedding_time_per_fold_s" : emb_time_per_fold,
        "peak_gpu_mem_embedding_mb" : gpu_mem_emb_mb,
        "paradigm"                  : "hybrid (frozen embedding only, no linguistic features)",
    }
    joblib_path = os.path.join(CFG["output_dir"], "experiment_results_no_fe.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df[cols_print].to_csv(os.path.join(CFG["output_dir"], "cv_results_indobert_hyb_no_fe.csv"), index=False)
    all_preds_df.to_csv(os.path.join(CFG["output_dir"], "all_predictions_indobert_hyb_no_fe.csv"), index=False)

    print(f"\n  Hasil lengkap disimpan: {joblib_path}")
    return results_df


if __name__ == "__main__":
    main()


---


In [ ]:
"""
=============================================================
  AES — Paradigma B: DistilBERT-Indo Hybrid (tanpa fitur linguistik)
  Metode   : Frozen embedding + XGBoost  (TANPA Feature Engineering)
  Dataset  : preprocessed_texts.pkl, scores.pkl  (SAMA dengan versi +FE)
  Model    : cahya/distilbert-base-indonesian
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation (KFold)
  Referensi: Faseeh et al. (2024), Chen & Guestrin (2016)
=============================================================
CATATAN ABLASI:
  - Satu-satunya perbedaan dengan cell "DistilBERT-Indo Hybrid":
    X_all HANYA berisi embedding frozen ([CLS], 768 dim), TIDAK
    digabung dengan 6 fitur linguistik.
  - Dataset, n_folds, seed, hyperparameter XGBoost dibuat identik
    dengan versi +FE agar selisih QWK/MSE dapat diatribusikan
    pada fitur linguistik, bukan pada confound lain.

=============================================================
"""

import os
import time
import threading
import traceback
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import pynvml


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay"        : f"{ESSAY_PATH}preprocessed_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",
    "model_name"   : "cahya/distilbert-base-indonesian",
    "max_len"      : 256,
    "embed_dim"    : 768,
    "batch_size"   : 32,
    "xgb_n_estimators"     : 300,
    "xgb_learning_rate"    : 0.05,
    "xgb_max_depth"        : 3,
    "xgb_subsample"        : 0.5,
    "xgb_colsample_bytree" : 0.5,
    "xgb_reg_lambda"       : 2.0,
    "xgb_reg_alpha"        : 0.1,
    "xgb_gamma"            : 0.1,
    "xgb_min_child_weight" : 20,
    "xgb_early_stop"       : 20,
    "n_folds"      : 10,
    "seed"         : 42,
    "output_dir"   : f"{OUTPUT_PATH}Hasil eksperiment/output_distilbert_hyb_no_fe",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {DEVICE}")
print(f"Model          : {CFG['model_name']} (FROZEN)")
print(f"Fitur tambahan : TIDAK DIGUNAKAN (embedding only)")


# ══════════════════════════════════════════════════════════
#  PYNVML — polling peak GPU untuk XGBoost (non-PyTorch)
# ══════════════════════════════════════════════════════════
def _gpu_poll_worker(stop_event, peak_holder, device_index=0):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(device_index)
    peak = 0
    while not stop_event.is_set():
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        if info.used > peak:
            peak = info.used
        stop_event.wait(timeout=0.05)
    peak_holder.append(peak)
    pynvml.nvmlShutdown()


def measure_gpu_peak_mb(fn, device_index=0):
    stop_event  = threading.Event()
    peak_holder = []
    t = threading.Thread(target=_gpu_poll_worker,
                         args=(stop_event, peak_holder, device_index), daemon=True)
    t.start()
    result = fn()
    stop_event.set()
    t.join()
    peak_mb = peak_holder[0] / (1024 ** 2) if peak_holder else 0.0
    return result, peak_mb


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating=None, max_rating=None):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)

    if min_rating is None:
        min_rating = min(y_true.min(), y_pred.min())
    if max_rating is None:
        max_rating = max(y_true.max(), y_pred.max())

    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1

    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored

    weights = np.zeros((num_ratings, num_ratings))
    for i in range(num_ratings):
        for j in range(num_ratings):
            weights[i][j] = ((i - j) ** 2) / ((num_ratings - 1) ** 2)

    numerator   = np.sum(weights * conf_mat)
    denominator = np.sum(weights * expected)
    if denominator == 0:
        return 0.0
    return 1.0 - numerator / denominator


# ══════════════════════════════════════════════════════════
#  DATASET — untuk ekstraksi embedding saja
# ══════════════════════════════════════════════════════════
class EssayEmbedDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }


# ══════════════════════════════════════════════════════════
#  FROZEN EXTRACTOR
# ══════════════════════════════════════════════════════════
class DistilBERTExtractor(nn.Module):
    """
    Frozen embedding extractor.
    Bobot tidak pernah diperbarui — hanya untuk ekstraksi representasi teks.
    Representasi via [CLS] token, output dim 768.
    """
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        emb = out.last_hidden_state[:, 0, :]   # [CLS] token, dim 768
        return emb


def extract_embeddings(texts, tokenizer, model, cfg, device):
    """
    Ekstrak embedding dari seluruh dataset dalam satu pass.
    GPU memory diukur dengan torch.cuda.max_memory_allocated().
    Waktu diukur dengan time.perf_counter().

    Returns:
        embeddings      : np.ndarray (N, embed_dim)
        t_emb_total     : float, waktu ekstraksi dalam detik
        gpu_mem_emb_mb  : float, peak GPU memory saat ekstraksi dalam MB
    """
    model.eval()
    dataset  = EssayEmbedDataset(texts, tokenizer, cfg["max_len"])
    loader   = DataLoader(dataset, batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)
    all_embs = []

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    t_emb_start = time.perf_counter()

    with torch.no_grad():
        for batch in loader:
            embs = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
            )
            all_embs.append(embs.cpu().numpy())

    t_emb_total = time.perf_counter() - t_emb_start

    if device.type == "cuda":
        gpu_mem_emb_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    else:
        gpu_mem_emb_mb = 0.0

    return np.vstack(all_embs), t_emb_total, gpu_mem_emb_mb


# ══════════════════════════════════════════════════════════
#  XGBOOST — STATIC HYPERPARAMETER
# ══════════════════════════════════════════════════════════
def train_xgboost_static(X_train, y_train, X_val, y_val, cfg):
    params = {
        "n_estimators"    : cfg["xgb_n_estimators"],
        "learning_rate"   : cfg["xgb_learning_rate"],
        "max_depth"       : cfg["xgb_max_depth"],
        "subsample"       : cfg["xgb_subsample"],
        "colsample_bytree": cfg["xgb_colsample_bytree"],
        "booster"         : "gbtree",
        "reg_lambda"      : cfg["xgb_reg_lambda"],
        "reg_alpha"       : cfg["xgb_reg_alpha"],
        "gamma"           : cfg["xgb_gamma"],
        "min_child_weight": cfg["xgb_min_child_weight"],
        "tree_method"     : "hist",
        "device"          : "cuda",
        "random_state"    : cfg["seed"],
        "verbosity"       : 0,
    }

    model = xgb.XGBRegressor(
        **params,
        early_stopping_rounds=cfg["xgb_early_stop"]
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    return model, params


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx,
             X_all, y_all, cfg, device,
             emb_time_per_fold, es_idx=None):
    """
    Parameters
    ----------
    emb_time_per_fold : float
        t_emb_total / n_folds — porsi embedding extraction time untuk fold ini.
    """
    print(f"\n{'='*55}")
    print(f"  FOLD {fold_idx + 1}/{cfg['n_folds']}")
    print(f"{'='*55}")

    X_train = X_all[train_idx]
    X_val   = X_all[val_idx]
    y_train = y_all[train_idx]
    y_val   = y_all[val_idx]
    X_es, y_es = X_all[es_idx], y_all[es_idx]   # [FIX-LEAKAGE] early-stopping set

    print(f"  Train: {len(X_train)} | Val: {len(X_val)}")
    print(f"  Input dim: {X_train.shape[1]} ({cfg['embed_dim']} embed, tanpa FE)")
    print("  Training XGBoost dengan hyperparameter statis...")

    gpu_index = device.index if (device.type == "cuda" and device.index is not None) else 0

    t_xgb_start = time.perf_counter()
    (final_model, best_params), xgb_gpu_peak_mb = measure_gpu_peak_mb(
        lambda: train_xgboost_static(X_train, y_train, X_es, y_es, cfg),
        device_index=gpu_index,
    )
    t_xgb_train = time.perf_counter() - t_xgb_start

    total_training_time = emb_time_per_fold + t_xgb_train

    # ── Inference time per sampel (warm-up + pengukuran) ──
    _ = final_model.predict(X_val[:1])   # warm-up
    t_infer_start = time.perf_counter()
    _ = final_model.predict(X_val[:1])
    inference_time_ms = (time.perf_counter() - t_infer_start) * 1000

    # ── Prediksi seluruh validation set untuk metrik ──
    preds_val = final_model.predict(X_val)
    mse_val   = mean_squared_error(y_val, preds_val)
    qwk_val   = quadratic_weighted_kappa(y_val, preds_val, cfg["score_min"], cfg["score_max"])

    print(f"\n  → Fold {fold_idx+1} FINAL:")
    print(f"    QWK                    = {qwk_val:.4f}")
    print(f"    MSE                    = {mse_val:.4f}")
    print(f"    training_time_s        = {total_training_time:.2f}s "
          f"(emb_share={emb_time_per_fold:.2f}s + xgb={t_xgb_train:.2f}s)")
    print(f"    inference_ms           = {inference_time_ms:.4f} ms/sampel")
    print(f"    memory_mb [xgb_gpu]    = {xgb_gpu_peak_mb:.2f} MB")

    fold_preds_df = pd.DataFrame({
        "true_score": y_val,
        "pred_score": preds_val,
        "fold"      : fold_idx + 1,
    })

    result = {
        "fold"              : fold_idx + 1,
        "qwk"               : qwk_val,
        "mse"               : mse_val,
        "training_time_s"   : total_training_time,
        "inference_ms"      : inference_time_ms,
        "memory_mb_xgb_gpu" : xgb_gpu_peak_mb,
        "best_params"       : best_params,
    }
    return result, fold_preds_df, final_model


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    np.random.seed(CFG["seed"])

    # ── 1. Load data ──────────────────────────────────────
    print("\n[1/4] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])

    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())

    print(f"  Total sampel : {len(texts)}")
    print(f"  Rentang skor : {CFG['score_min']} – {CFG['score_max']}")
    print(f"  Rata-rata    : {scores.mean():.2f} ± {scores.std():.2f}")

    # ── 2. Ekstrak embedding (SATU KALI, frozen) ──────────
    print(f"\n[2/4] Mengekstrak frozen embedding: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
    extractor = DistilBERTExtractor(CFG["model_name"])
    extractor.to(DEVICE)
    extractor.eval()

    embeddings, t_emb_total, gpu_mem_emb_mb = extract_embeddings(
        texts, tokenizer, extractor, CFG, DEVICE
    )

    emb_time_per_fold = t_emb_total / CFG["n_folds"]

    print(f"  Embedding shape        : {embeddings.shape}")
    print(f"  Waktu ekstraksi total  : {t_emb_total:.2f}s")
    print(f"  Waktu per fold (share) : {emb_time_per_fold:.2f}s")
    print(f"  Peak GPU memory (emb)  : {gpu_mem_emb_mb:.2f} MB")

    emb_path = os.path.join(CFG["output_dir"], "embeddings_distilbert_hyb_no_fe.npy")
    np.save(emb_path, embeddings)
    print(f"  Embedding disimpan: {emb_path}")

    # ── 3. TANPA fitur linguistik — X_all = embedding saja ─
    X_all = embeddings.astype(np.float32)
    y_all = scores.astype(np.float32)
    print(f"\n  Input matrix X_all: {X_all.shape}")
    print(f"  ({CFG['embed_dim']} embedding dim, tanpa FE)")

    # ── 4. Cross-Validation ───────────────────────────────
    print(f"\n[3/4] Memulai {CFG['n_folds']}-fold cross-validation...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])

    all_results   = []
    all_preds_dfs = []
    fold_models   = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all)):

        train_idx, es_idx = inner_split(train_idx, y_all, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df, fold_model = run_fold(
                fold_idx, train_idx, val_idx,
                X_all, y_all, CFG, DEVICE,
                emb_time_per_fold=emb_time_per_fold, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
            fold_models.append(fold_model)

            model_path = os.path.join(
                CFG["output_dir"],
                f"xgb_model_fold{fold_idx+1}_distilbert_hyb_no_fe.json"
            )
            fold_model.save_model(model_path)
            print(f"  Model fold {fold_idx+1} disimpan: {model_path}")

        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1} gagal: {e}")
            traceback.print_exc()

    # ── 5. Ringkasan & Simpan ─────────────────────────────
    print(f"\n[4/4] Menyimpan hasil...")
    print(f"\n{'='*55}")
    print(f"  RINGKASAN HASIL — DistilBERT-Indo Hybrid (tanpa fitur linguistik)")
    print(f"{'='*55}")

    results_df   = pd.DataFrame(all_results)
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)

    cols_print = [c for c in results_df.columns if c != 'best_params']
    print(results_df[cols_print].to_string(index=False))

    print(f"\n  Rata-rata ± Std:")
    for metric in ["qwk", "mse", "training_time_s", "inference_ms", "memory_mb_xgb_gpu"]:
        m = results_df[metric].mean()
        s = results_df[metric].std()
        print(f"    {metric:<28s}: {m:.4f} ± {s:.4f}")

    print(f"\n  Embedding extraction (global):")
    print(f"    peak_gpu_mem_emb_mb : {gpu_mem_emb_mb:.2f} MB")

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"],
    )
    overall_mse = mean_squared_error(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values,
    )
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")

    experiment_data = {
        "config"                       : CFG,
        "fold_results"                 : all_results,
        "fold_predictions"             : all_preds_df,
        "overall_qwk"                  : overall_qwk,
        "overall_mse"                  : overall_mse,
        "timestamp"                    : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "feature_names"                : None,
        "embed_shape"                  : embeddings.shape,
        "embedding_extraction_time_s"  : t_emb_total,
        "embedding_time_per_fold_s"    : emb_time_per_fold,
        "peak_gpu_mem_embedding_mb"    : gpu_mem_emb_mb,
        "paradigm"                     : "hybrid (frozen embedding only, no linguistic features)",
    }
    joblib_path = os.path.join(CFG["output_dir"], "experiment_results_no_fe.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df[cols_print].to_csv(
        os.path.join(CFG["output_dir"], "cv_results_distilbert_hyb_no_fe.csv"),
        index=False,
    )
    all_preds_df.to_csv(
        os.path.join(CFG["output_dir"], "all_predictions_distilbert_hyb_no_fe.csv"),
        index=False,
    )

    print(f"\n  Hasil lengkap disimpan: {joblib_path}")
    print(f"  Output direktori      : {CFG['output_dir']}/")

    return results_df


if __name__ == "__main__":
    main()
